## Example 1 — a BF16 GEMM compiled on first use and checked against torch

In [ ]:
# DeepGEMM compiles this kernel at run time, so the cell also needs the CUDA toolkit (nvcc and the CUDA headers).
import os, tempfile, torch
assert torch.cuda.is_available(), 'requires-gpu cell ran but torch sees no CUDA device'
os.environ['DG_JIT_CACHE_DIR'] = tempfile.mkdtemp(prefix='deep_gemm_nb_')  # before the first kernel
import deep_gemm
cc = torch.cuda.get_device_capability()
if cc[0] not in (9, 10, 12):
    print(f'SKIP: DeepGEMM has no kernels for sm_{cc[0]}{cc[1]} (SM90, SM100 and SM120 families only)')
else:
    a = torch.randn(256, 1024, device='cuda', dtype=torch.bfloat16)
    b = torch.randn(512, 1024, device='cuda', dtype=torch.bfloat16)
    d = torch.empty(256, 512, device='cuda', dtype=torch.bfloat16)
    # Without this, bf16_gemm_nt takes cuBLASLt and compiles no kernel (csrc/apis/gemm.hpp, bf16_gemm_nt).
    deep_gemm.use_deterministic_algorithms(True)
    deep_gemm.bf16_gemm_nt(a, b, d)
    ref = a.float() @ b.float().t()
    # calc_diff, adapted from DeepGEMM's tests (MIT): 0 when d equals ref, growing as they diverge.
    diff = 1 - 2 * (d.double() * ref.double()).sum() / (d.double() ** 2 + ref.double() ** 2).sum()
    assert diff.item() < 1e-5, diff.item()
    print(f'OK -- bf16_gemm_nt matches torch (calc_diff {diff.item():.2e})')

## Example 2 — the cuBLASLt path, and a bad shape rejected with an error rather than a crash

In [ ]:
import torch, deep_gemm
assert torch.cuda.is_available(), 'requires-gpu cell ran but torch sees no CUDA device'
a = torch.randn(128, 256, device='cuda', dtype=torch.bfloat16)
b = torch.randn(64, 256, device='cuda', dtype=torch.bfloat16)
d = torch.empty(128, 64, device='cuda', dtype=torch.bfloat16)
deep_gemm.cublaslt_gemm_nt(a, b, d)  # _C calls cuBLASLt itself, so this proves it is linked
torch.testing.assert_close(d.float(), a.float() @ b.float().t(), rtol=2e-2, atol=2e-1)
try:
    deep_gemm.cublaslt_gemm_nt(a, b[:, :128].contiguous(), d)  # K does not match
    raise AssertionError('a mismatched K was accepted')
except RuntimeError as error:
    print('OK -- cuBLASLt GEMM matches torch; a mismatched K is refused:', str(error).splitlines()[0][:120])
